# Dossier économétrie
*Younes, Yale, Samuel, Manon*

Dans la première partie, nous avons nettoyé les bases trainpol, trainclaim et DataClimatiques et construit deux bases principales :
* base des assurés,
* une base des sinistres.

Ces deux bases contiennent donc :
* des informations descriptives sur les assurés (âge, sexe, profession, etc.),
* des caractéristiques de leurs véhicules,
* ainsi que le nombre et le coût des sinistres enregistrés.

Nous passons à l'analyse économétrique, dans les objectifs de mieux comprendre les facteurs associés au risque de sinistre en identifiant ses variables explicatives, estimer sa probabilité et modéliser la fréquence et le coût des sinistres.

Nous utiliserons des modèles économétriques classiques tels que :
* la régression logistique pour la sinistralité (variable binaire),
* l'ACP et l'ACM pour identifier des variables explicatives et des profils d'assurés / sinistrés,
* des modèles linéaires généralisés (GLM) pour modéliser les montants de sinistres. 

Ces résultats soutiendront le projet de machine learning, en affinant les critères de segmentation des profils assurés.

In [1]:
import numpy as np, pandas as pd, matplotlib as plt

In [ ]:
base_ass = pd.read_csv("base_ass.csv") # base des assurés
base_sin = pd.read_csv("base_sin.csv") # base des sinistrés 
base_non_sin = base_ass[~base_ass.idpolicy.isin(base_sin['idpolicy'])] # extrapoler la base des non sinistrés

La fréquence de sinistre de notre bas de données est d'environ 15 %

Déterminons le cout moyen de sinistre 

In [ ]:
cm = sum(base_ass.claim_amount) / sum(base_ass.nb_sin)
cm

869.7433373590983

Le cout moyen de sinistre est de 909.8515 euros environ

Déterminons le montant d'une eventuelle prime 

In [ ]:
pp = freq*cm
pp

133.71970723857274

La prime de sinistre basique est d'environ 131.3538 euros 

In [ ]:
base_ass.nb_sin.value_counts(dropna=False)

claim_nb
0.0    28146
1.0     4749
Name: count, dtype: int64

## Statistiques Descriptives

### Analyses univariées

Nous allons commencer par créer une **fonction qui produit automatiquement les analyses univariées** selon le type de données.  
Nous le ferons sur la **base des assurés**, puis sur celle des **sinistrés**, et sur une base des **non sinistrés** pour commencer à identifier des profils différents.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

  """
   Nous analysons les variables de chaque base de données.
    - Pour les variables quantitatives : statistiques descriptives et histogramme.
    - Pour les variables qualitatives : répartition des modalités et diagramme en barres.
    """


In [ ]:
def analyser_variables(df):
   for col in df.columns:
        print(f"Analysons la variable : {col}")
        print("-" * 50)
        
        # Nombre de valeurs manquantes
        missing_values = df[col].isnull().sum()
        print(f"Valeurs manquantes : {missing_values}")
        
        # Calcul du mode
        mode = df[col].mode().iloc[0] if not df[col].mode().empty else "Pas de mode"
        print(f"Mode : {mode}")
        
        # Vérifier si la variable est quantitative
        if pd.api.types.is_numeric_dtype(df[col]):
            print(df[col].describe())
            plt.figure(figsize=(6, 4))
            sns.histplot(df[col], kde=True, bins=20, color="skyblue")
            plt.title(f"Histogramme de {col}")
            plt.xlabel(col)
            plt.ylabel("Fréquence")
            plt.show()
        
        # Vérifier si la variable est qualitative
        else:
            print(df[col].value_counts())
            unique_values = df[col].nunique()
            
            # Diagramme circulaire pour les variables avec deux modalités
            if unique_values == 2:
                plt.figure(figsize=(6, 6))
                df[col].value_counts().plot.pie(autopct='%1.1f%%', colors=["lightblue", "lightgreen"])
                plt.title(f"Diagramme circulaire de {col}")
                plt.ylabel("")  # Supprimer le label inutile
                plt.show()
            
            # Diagramme en barres pour les autres variables qualitatives
            else:
                plt.figure(figsize=(6, 4))
                sns.countplot(x=df[col], order=df[col].value_counts().index, palette="viridis")
                plt.title(f"Diagramme en barres de {col}")
                plt.xlabel(col)
                plt.ylabel("Nombre d'occurrences")
                plt.xticks(rotation=45)
                plt.show()

Analyse de la base des assurés

In [ ]:
analyser_variables(base_ass)

Analyse de la base des sinistrés

In [ ]:
analyser_variables(base_sin)

### Analyses multivariées

L'objectif ici est de commencer à identifier des corrélations entre les variables:  
D'une part entre les variables descriptives pour identifier celles qui pourraient être correlées et donc utiles pour identifier un profil d'assuré, mais à ne pas utiliser en double pour la modélisation,  
D'autre part entre ces variables et la sinistralité, afin d'établir déjà quelles variables peuvent avoir un impact plus important sur celle ci. 

Pour cela, nous allons créer une fonction qui renvoie pour chaque variable sa corrélation avec les autres de la base, trié du plus correlé au moins correlé, de façon à commencer à identifier des dépendances. 

#### Fonction 1

In [ ]:
def analyse_multivariee(df):
# Variables quantitatives
    quanti_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if len(quanti_cols) > 1:
        print("Analyse des variables quantitatives:")
        correlation_matrix = df[quanti_cols].corr()
        print("Matrice de corrélation :")
        print(correlation_matrix)

        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
        plt.title("Matrice de corrélation (Heatmap)")
        plt.show()

    else:
        print("Pas assez de variables quantitatives pour effectuer une matrice de corrélation.")

    # Variables qualitatives
    quali_cols = [col for col in df.columns if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype == 'object']
    if len(quali_cols) > 1:
        print("Analyse des variables qualitatives :")
        for i, col1 in enumerate(quali_cols):
            for col2 in quali_cols[i+1:]:
                print(f"Tableau croisé entre {col1} et {col2} :")
                crosstab = pd.crosstab(df[col1], df[col2])
                print(crosstab)

                plt.figure(figsize=(8, 6))
                sns.heatmap(crosstab, annot=True, cmap="viridis", fmt="d", cbar=False)
                plt.title(f"Heatmap des relations entre {col1} et {col2}")
                plt.show()
    else:
        print("Pas assez de variables qualitatives pour effectuer une analyse croisée.")

Analyse multivarié de la base des assurés

In [ ]:
analyse_multivariee(base_ass)

Analyse multivarié de la base des sinistrés

In [ ]:
analyse_multivariee(base_sin)

#### Fonction 2

In [ ]:
import pandas as pd

def Analyse_multi(df, cible):
    """
        df: Base de données contenant les variables.
        cible (str): Nom de la colonne représentant la sinistralité.
    """
    correlations = {}
    for col in df.columns:
        if col != cible and pd.api.types.is_numeric_dtype(df[col]):
            corr_value = df[col].corr(df[cible])
            correlations[col] = corr_value

    # Tri des corrélations par ordre décroissant
    sorted_correlations = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    # Retourner les corrélations sous forme de DataFrame
    result = pd.DataFrame(sorted_correlations, columns=["Variable", "Corrélation avec Sinistralité"])
    return result

Analyse multivarié de la base des assurés

In [ ]:
Analyse_multi(base_ass, cible)

Analyse multivarié de la base des sinistrés

In [ ]:
Analyse_multi(base_sin, cible)

## Conclusions

xxx